Purpose of program:
    create and maintain DB that centralizes links to useful information sources
        websites, podcasts (yt), yt vids, online archives, offline files

Categories:
    current events
    historical docs
    entertainment
    scientific research
    education - philosophy, history, IT

Output:
    add and remove urls via webpage interface
    display organized lists of urls and thumbnails
    filtering of lists based on:
        date range
        type of content - vid, ebook, webpage
        source of content - yt, website
        subject matter - categories

Questions:
    link to chrome bookmarks

In [ ]:
import os
import subprocess

subprocess.Popen(
    [r"C:\Program Files\Anvsoft\Any Video Converter\Any Video Converter.exe"]
)

<Popen: returncode: None args: ['C:\\Program Files\\Anvsoft\\Any Video Conve...>

In [ ]:
import subprocess

try:
    result = subprocess.run(
        [r"C:\Program Files\Anvsoft\Any Video Converter\Any Video Converter.exe"],
        capture_output=True,
        text=True,
    )
    print("stdout:", result.stdout)
    print("stderr:", result.stderr)
    print("returncode:", result.returncode)
except Exception as e:
    print("Error:", e)

In [ ]:
import subprocess

input_file = r"C:\path\to\your\video.mp4"
output_file = r"C:\path\to\output.avi"

subprocess.run(["ffmpeg", "-i", input_file, output_file])

In [ ]:
import re

numb = "123abc456def789ghi"
per = "1755-1789"
numbers = re.search(r"\d+", numb)
print(numbers)  # Output: ['123', '456', '789']

<re.Match object; span=(0, 3), match='123'>


In [ ]:
# check contents of db

import sqlite3

conn = sqlite3.connect(r"C:\Users\Ken\Python files\Link-DB\instance\links.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
cursor.execute("SELECT * FROM history_page")
rows = cursor.fetchall()
for row in rows:
    print(dict(row))
conn.close()

In [ ]:
import sqlite3
from datetime import datetime, timezone

SQLITE_PATH = r"C:\Users\Ken\Python files\Link-DB\instance\links.db"

existing_pages = [
    {
        "title": "The American Revolution",
        "slug": "the-american-revolution",
        "period": "1765-1783",
    },
    {
        "title": "The French Revolution",
        "slug": "the-french-revolution",
        "period": "1789-1799",
    },
    {
        "title": "The Napoleonic Wars",
        "slug": "the-napoleonic-wars",
        "period": "1803-1815",
    },
]

conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()

now = datetime.now(timezone.utc).isoformat()

for page in existing_pages:
    cursor.execute("SELECT id FROM history_page WHERE slug = ?", (page["slug"],))
    if not cursor.fetchone():
        cursor.execute(
            """
            INSERT INTO history_page (title, slug, period, date_added, last_modified)
            VALUES (?, ?, ?, ?, ?)
        """,
            (page["title"], page["slug"], page["period"], now, now),
        )
        print(f"Added: {page['title']}")
    else:
        print(f"Already exists: {page['title']}")

conn.commit()
conn.close()
print("Done!")

Added: The American Revolution
Added: The French Revolution
Added: The Napoleonic Wars
Done!


In [2]:
import sqlite3

conn = sqlite3.connect(r"C:\Users\Ken\Python files\Link-DB\instance\links.db")
cursor = conn.cursor()

columns = [
    "ALTER TABLE history_page ADD COLUMN era VARCHAR(200)",
    "ALTER TABLE history_page ADD COLUMN period VARCHAR(200)",
    "ALTER TABLE history_page ADD COLUMN phase VARCHAR(200)",
    "ALTER TABLE history_page ADD COLUMN start_year INTEGER",
]

for sql in columns:
    try:
        cursor.execute(sql)
        print(f"Added: {sql.split('COLUMN')[1].strip()}")
    except Exception as e:
        print(f"Skipped: {e}")

conn.commit()
conn.close()
print("Done!")

Skipped: duplicate column name: era
Skipped: duplicate column name: period
Added: phase VARCHAR(200)
Added: start_year INTEGER
Done!


In [3]:
import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(os.getenv("RENDER_DATABASE_URL"))
cursor = conn.cursor()

columns = [
    "ALTER TABLE history_page ADD COLUMN era VARCHAR(200)",
    "ALTER TABLE history_page ADD COLUMN period VARCHAR(200)",
    "ALTER TABLE history_page ADD COLUMN phase VARCHAR(200)",
    "ALTER TABLE history_page ADD COLUMN start_year INTEGER",
]

for sql in columns:
    try:
        cursor.execute(sql)
        print(f"Added: {sql.split('COLUMN')[1].strip()}")
    except Exception as e:
        conn.rollback()
        print(f"Skipped: {e}")

conn.commit()
conn.close()
print("Done!")

Added: era VARCHAR(200)
Skipped: column "period" of relation "history_page" already exists

Added: phase VARCHAR(200)
Added: start_year INTEGER
Done!


In [4]:
import sqlite3

conn = sqlite3.connect(r"C:\Users\Ken\Python files\Link-DB\instance\links.db")
cursor = conn.cursor()

# move existing period value into era, clear period
cursor.execute("UPDATE history_page SET era = period, period = NULL")

conn.commit()
conn.close()
print("Done!")

Done!


In [ ]:
import sqlite3
from datetime import datetime, timezone

SQLITE_PATH = r"C:\Users\Ken\Python files\Link-DB\instance\links.db"

test_pages = [
    {
        "title": "The Bronze Age Collapse",
        "slug": "the-bronze-age-collapse",
        "era": "Ancient History",
        "period": "Early Civilizations",
        "phase": "Late Bronze Age",
        "start_year": -1200,
    },
    {
        "title": "The Fall of Rome",
        "slug": "the-fall-of-rome",
        "era": "Ancient History",
        "period": "Roman Era",
        "phase": "Late Roman Empire",
        "start_year": 476,
    },
    {
        "title": "The Black Death",
        "slug": "the-black-death",
        "era": "Middle Ages",
        "period": "Late Middle Ages",
        "phase": "Black Death Phase",
        "start_year": 1347,
    },
    {
        "title": "The Cold War",
        "slug": "the-cold-war",
        "era": "Modern Period",
        "period": "Cold War",
        "phase": "Early Cold War",
        "start_year": 1947,
    },
]

conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()
now = datetime.now(timezone.utc).isoformat()

for page in test_pages:
    cursor.execute("SELECT id FROM history_page WHERE slug = ?", (page["slug"],))
    if not cursor.fetchone():
        cursor.execute(
            """
            INSERT INTO history_page (title, slug, era, period, phase, start_year, date_added, last_modified)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
            (
                page["title"],
                page["slug"],
                page["era"],
                page["period"],
                page["phase"],
                page["start_year"],
                now,
                now,
            ),
        )
        print(f"Added: {page['title']}")
    else:
        print(f"Already exists: {page['title']}")

conn.commit()
conn.close()
print("Done!")

Added: The Bronze Age Collapse
Added: The Fall of Rome
Added: The Black Death
Added: The Cold War
Done!


In [ ]:
import sqlite3
from datetime import datetime, timezone

SQLITE_PATH = r"C:\Users\Ken\Python files\Link-DB\instance\links.db"

new_pages = [
    {
        "title": "The Crisis of the Third Century",
        "slug": "the-crisis-of-the-third-century",
        "era": "Ancient History",
        "period": "Roman Era",
        "phase": "Late Roman Empire",
        "start_year": 235,
    },
    {
        "title": "The Sack of Rome",
        "slug": "the-sack-of-rome",
        "era": "Ancient History",
        "period": "Roman Era",
        "phase": "Late Roman Empire",
        "start_year": 410,
    },
]

conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()
now = datetime.now(timezone.utc).isoformat()

for page in new_pages:
    cursor.execute("SELECT id FROM history_page WHERE slug = ?", (page["slug"],))
    if not cursor.fetchone():
        cursor.execute(
            """
            INSERT INTO history_page (title, slug, era, period, phase, start_year, date_added, last_modified)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
            (
                page["title"],
                page["slug"],
                page["era"],
                page["period"],
                page["phase"],
                page["start_year"],
                now,
                now,
            ),
        )
        print(f"Added: {page['title']}")
    else:
        print(f"Already exists: {page['title']}")

conn.commit()
conn.close()
print("Done!")

Added: The Crisis of the Third Century
Added: The Sack of Rome
Done!


In [1]:
import os

pages = [
    "the-crisis-of-the-third-century",
    "the-sack-of-rome",
]

template = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>{title}</title>
  <link rel="stylesheet" href="/static/darkstyle.css">
</head>
<body>
  <h1>{title}</h1>
  <a href="/history" class="back-link">← Back History</a>
  <hr>
  <p>No content yet.</p>
</body>
</html>"""

for slug in pages:
    title = slug.replace("-", " ").title()
    filepath = os.path.join(
        r"C:\Users\Ken\Python files\Link-DB\templates\history_pages", f"{slug}.html"
    )
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(template.format(title=title))
    print(f"Created: {filepath}")

Created: C:\Users\Ken\Python files\Link-DB\templates\history_pages\the-crisis-of-the-third-century.html
Created: C:\Users\Ken\Python files\Link-DB\templates\history_pages\the-sack-of-rome.html


In [1]:
import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(os.getenv("RENDER_DATABASE_URL"))
cursor = conn.cursor()

columns = [
    "ALTER TABLE history_page ADD COLUMN era VARCHAR(200)",
    "ALTER TABLE history_page ADD COLUMN phase VARCHAR(200)",
    "ALTER TABLE history_page ADD COLUMN start_year INTEGER",
]

for sql in columns:
    try:
        cursor.execute(sql)
        print(f"Added: {sql.split('COLUMN')[1].strip()}")
    except Exception as e:
        conn.rollback()
        print(f"Skipped: {e}")

conn.commit()
conn.close()
print("Done!")

Added: era VARCHAR(200)
Skipped: column "phase" of relation "history_page" already exists

Skipped: column "start_year" of relation "history_page" already exists

Done!


In [4]:
import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(os.getenv("RENDER_DATABASE_URL"))
cursor = conn.cursor()
cursor.execute("""
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_name = 'history_page'
    ORDER BY ordinal_position
""")
cols = cursor.fetchall()
for col in cols:
    print(col[0])
conn.close()

id
title
slug
period
date_added
last_modified
phase
start_year
era


In [3]:
import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(os.getenv("RENDER_DATABASE_URL"))
cursor = conn.cursor()
cursor.execute("ALTER TABLE history_page ADD COLUMN era VARCHAR(200)")
conn.commit()
conn.close()
print("Done!")

Done!
